In [233]:
import pandas as pd
import numpy as np

# load raw dataset
raw = pd.read_csv("../data/raw/DP_LIVE_02012022050459635.csv")
raw.head()

,LOCATION,INDICATOR,SUBJECT,MEASURE,FREQUENCY,TIME,Value,Flag Codes
0,AUS,OILPROD,TOT,KTOE,A,1960,NaN,L
1,AUS,OILPROD,TOT,KTOE,A,1961,NaN,L
2,AUS,OILPROD,TOT,KTOE,A,1962,NaN,L
3,AUS,OILPROD,TOT,KTOE,A,1963,NaN,L
4,AUS,OILPROD,TOT,KTOE,A,1964,NaN,L


In [234]:
#Drop Unnecessary Columns
df = raw.drop(columns=[
    "INDICATOR",
    "SUBJECT",
    "MEASURE",
    "FREQUENCY",
    "Flag Codes"
])
df.head()

,LOCATION,TIME,Value
0,AUS,1960,NaN
1,AUS,1961,NaN
2,AUS,1962,NaN
3,AUS,1963,NaN
4,AUS,1964,NaN


In [235]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8236 entries, 0 to 8235
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   LOCATION  8236 non-null   object 
 1   TIME      8236 non-null   int64  
 2   Value     6104 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 193.2+ KB


,TIME,Value
count,8236.000000,6.104000e+03
mean,1988.500000,6.533908e+04
std,16.741685,3.259844e+05
min,1960.000000,0.000000e+00
25%,1974.000000,0.000000e+00
50%,1988.500000,6.620815e+02
75%,2003.000000,1.425258e+04
max,2017.000000,3.991795e+06


In [236]:
# Check for missing values
df.isnull().sum()

LOCATION       0
TIME           0
Value       2132
dtype: int64

In [237]:
# Handling missing values
df[df["Value"].isnull()].head()
df = df.dropna(subset=["Value"])
df.isnull().sum()

LOCATION    0
TIME        0
Value       0
dtype: int64

In [238]:
# Rename columns for better readability
df = df.rename(columns={
    "LOCATION": "country",
    "TIME": "year",
    "Value": "production"
})
df.head()

,country,year,production
11,AUS,1971,14226.194
12,AUS,1972,15029.094
13,AUS,1973,18720.577
14,AUS,1974,18498.696
15,AUS,1975,19736.070


In [246]:
# drop the aggregate rows that aren’t individual countries
exclude = ["WLD", "G20", "OECD"]
df = df[~df["country"].isin(exclude)]
df.reset_index(drop=True, inplace=True)

# quick sanity check
df["country"].unique()[:10], len(df)

(array(['AGO', 'ALB', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BEL',
        'BEN'], dtype=object),
 5565)

In [239]:
# Sort the Dataset
df = df.sort_values(["country","year"])

In [240]:
n_raw   = len(raw)
n_clean = len(df)
print(f"  Raw rows       : {n_raw:,}")
print(f"  After cleansing: {n_clean:,}  ({n_raw - n_clean:,} missing-value rows dropped)")
print(f"  Countries      : {df['country'].nunique()}")
print(f"  Year range     : {df['year'].min()} – {df['year'].max()}")


  Raw rows       : 8,236
  After cleansing: 6,104  (2,132 missing-value rows dropped)
  Countries      : 142
  Year range     : 1971 – 2017


In [241]:
# Create Lag Feature
df["lag1"]        = df.groupby("country")["production"].shift(1)
df["lag2"]        = df.groupby("country")["production"].shift(2)
df["lag3"]        = df.groupby("country")["production"].shift(3)
df["rolling_3"]   = df.groupby("country")["production"].transform(
                        lambda x: x.shift(1).rolling(3).mean())
df["country_code"] = df["country"].astype("category").cat.codes


df = df.dropna()

In [244]:
# Visualisation: top-10 producers (latest year available) ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

OUTPUT_DIR = "../output"

latest_year = df["year"].max()
top10 = (
    df[df["year"] == latest_year]
    .nlargest(10, "production")
    .sort_values("production")
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(top10["country"], top10["production"] / 1_000,
               color="#2E75B6", edgecolor="white")
ax.set_xlabel("Production (thousand KTOE)")
ax.set_title(f"Top 10 Oil Producers — {latest_year}", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.bar_label(bars, fmt="{:,.0f}", padding=4, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
viz_path = os.path.join(OUTPUT_DIR, "top10_producers.png")
plt.savefig(viz_path, dpi=150)
plt.close()
print(f"\n  Visualisation saved → {viz_path}")


  Visualisation saved → ../output\top10_producers.png


In [245]:
# Save Processed Data
df.to_csv("../data/processed/oil_production_features.csv", index=False)